# 🔬 Nail Disorder Detection — 5-Fold Cross-Validation
**Models:** MobileNetV2 · ResNet50 · EfficientNetB0 · Ensemble

**Strategy:** Stratified 5-Fold Cross-Validation on full dataset (4147 images, 6 classes)

> ⚡ **Enable GPU before running:** Runtime → Change runtime type → T4 GPU
>
> 🔄 **Disconnect-safe:** If Colab disconnects, re-run Cells 1–5 then Cell 7. Already-completed folds are automatically skipped.

---

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'Pillow', '--quiet'])

print("Drive mounted. Ready.")

## Cell 2 — Imports

In [ ]:
import os, gc, json, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score
)
from PIL import Image

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

## Cell 3 — Configuration

In [ ]:
CFG = {
    # Paths
    "dataset_path"    : "/content/drive/MyDrive/main nail dataset",
    "save_dir"        : "/content/drive/MyDrive/nail_kfold_results",

    # Image & Training
    "img_size"        : (224, 224),
    "batch_size"      : 16,
    "num_classes"     : 6,
    "n_folds"         : 5,
    "seed"            : 42,

    # Phase 1: Head only (backbone frozen)
    "phase1_epochs"   : 10,
    "phase1_lr"       : 1e-3,

    # Phase 2: Fine-tune top layers
    "phase2_epochs"   : 20,
    "phase2_lr"       : 1e-5,
    "unfreeze_layers" : 30,

    # Model Head
    "dropout_rate"    : 0.4,
    "dense_units"     : 256,

    # Class names — must match folder names exactly
    "class_names"     : [
        'Acral_Lentiginous_Melanoma',
        'Healthy_Nail',
        'Onychogryphosis',
        'blue_finger',
        'clubbing',
        'pitting'
    ]
}

os.makedirs(CFG["save_dir"], exist_ok=True)
print("Config ready.")
print("Save directory:", CFG["save_dir"])

## Cell 4 — Load All File Paths & Labels
Reads from the **original unsplit dataset** (`main nail dataset` folder on Drive).

In [ ]:
all_paths  = []
all_labels = []

label_map = {name: idx for idx, name in enumerate(CFG["class_names"])}

for class_name in CFG["class_names"]:
    class_dir = os.path.join(CFG["dataset_path"], class_name)
    if not os.path.isdir(class_dir):
        print(f"  WARNING: folder not found -> {class_dir}")
        continue
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        for fpath in glob.glob(os.path.join(class_dir, ext)):
            all_paths.append(fpath)
            all_labels.append(label_map[class_name])

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

print(f"Total images found : {len(all_paths)}")
print(f"\n{'Class':<35} {'Count':>6} {'%':>6}")
print("-" * 50)
unique, counts = np.unique(all_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CFG['class_names'][u]:<33} {c:>6}  {c/len(all_labels)*100:>5.1f}%")
print("-" * 50)
print(f"  {'TOTAL':<33} {len(all_paths):>6}  100.0%")

assert len(all_paths) > 0, "No images found! Check dataset_path in CFG."
print("\nDataset loaded successfully.")

## Cell 5 — Custom Data Generator
Loads images on-the-fly by file path — no need to load the full dataset into RAM.

In [ ]:
def load_image(path, img_size):
    """Load one image and return float32 array in [0, 1]."""
    img = Image.open(path).convert('RGB').resize(img_size)
    return np.array(img, dtype=np.float32) / 255.0


class FoldSequence(keras.utils.Sequence):
    """
    Keras Sequence that loads images by file path on demand.
    Applies augmentation for training folds, plain rescale for validation.
    """
    def __init__(self, paths, labels, batch_size, img_size,
                 augment=False, num_classes=6, seed=42):
        self.paths       = np.array(paths)
        self.labels      = np.array(labels)
        self.batch_size  = batch_size
        self.img_size    = img_size
        self.augment     = augment
        self.num_classes = num_classes
        self.seed        = seed
        self.indices     = np.arange(len(self.paths))

        if augment:
            self.datagen = ImageDataGenerator(
                horizontal_flip    = True,
                rotation_range     = 15,
                zoom_range         = 0.20,
                width_shift_range  = 0.10,
                height_shift_range = 0.10,
                brightness_range   = [0.8, 1.2],
                fill_mode          = 'nearest'
            )

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        imgs = np.stack([load_image(self.paths[i], self.img_size) for i in batch_idx])
        lbls = to_categorical(self.labels[batch_idx], self.num_classes)
        if self.augment:
            imgs = next(self.datagen.flow(imgs, batch_size=len(imgs),
                                          shuffle=False, seed=self.seed))
        return imgs, lbls

    def on_epoch_end(self):
        if self.augment:
            np.random.shuffle(self.indices)

print("FoldSequence class ready.")

## Cell 6 — Model Builder & Training Functions
All shared functions: `build_model`, `get_callbacks`, `train_one_model`, `evaluate_on_fold`.

In [ ]:
def build_model(model_name, cfg):
    """
    Builds transfer-learning model with frozen backbone + custom head.
    Supports: 'mobilenetv2' | 'resnet50' | 'efficientnetb0'
    Returns (model, backbone)
    """
    img_shape = (*cfg["img_size"], 3)

    if model_name == 'mobilenetv2':
        backbone   = keras.applications.MobileNetV2(
            input_shape=img_shape, include_top=False, weights='imagenet')
        preprocess = keras.applications.mobilenet_v2.preprocess_input
    elif model_name == 'resnet50':
        backbone   = keras.applications.ResNet50(
            input_shape=img_shape, include_top=False, weights='imagenet')
        preprocess = keras.applications.resnet50.preprocess_input
    elif model_name == 'efficientnetb0':
        backbone   = keras.applications.EfficientNetB0(
            input_shape=img_shape, include_top=False, weights='imagenet')
        preprocess = keras.applications.efficientnet.preprocess_input
    else:
        raise ValueError(f"Unknown model: {model_name}")

    backbone.trainable = False
    inputs  = keras.Input(shape=img_shape, name="input_image")
    x       = preprocess(inputs * 255.0)
    x       = backbone(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(cfg["dense_units"], activation='relu')(x)
    x       = layers.Dropout(cfg["dropout_rate"])(x)
    outputs = layers.Dense(cfg["num_classes"], activation='softmax')(x)
    model   = keras.Model(inputs, outputs, name=f"{model_name}_model")
    return model, backbone


def get_callbacks(model_name, phase, fold, cfg):
    """EarlyStopping + ModelCheckpoint + ReduceLROnPlateau."""
    save_path = os.path.join(
        cfg["save_dir"],
        f"{model_name}_fold{fold}_phase{phase}_best.keras"
    )
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5,
            restore_best_weights=True, verbose=0
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=save_path, monitor='val_accuracy',
            save_best_only=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-7, verbose=0
        )
    ]


def train_one_model(model_name, fold, train_seq, val_seq,
                    class_weight_dict, cfg):
    """Full Phase 1 + Phase 2 training for one (model, fold) pair."""
    model, backbone = build_model(model_name, cfg)

    # Phase 1: frozen backbone
    print(f"    Phase 1 - frozen backbone")
    model.compile(
        optimizer=keras.optimizers.Adam(cfg["phase1_lr"]),
        loss='categorical_crossentropy', metrics=['accuracy']
    )
    model.fit(
        train_seq, epochs=cfg["phase1_epochs"],
        validation_data=val_seq,
        callbacks=get_callbacks(model_name, 1, fold, cfg),
        class_weight=class_weight_dict, verbose=1
    )

    # Phase 2: unfreeze top layers
    print(f"    Phase 2 - fine-tuning top {cfg['unfreeze_layers']} layers")
    backbone.trainable = True
    for layer in backbone.layers[:-cfg["unfreeze_layers"]]:
        layer.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(cfg["phase2_lr"]),
        loss='categorical_crossentropy', metrics=['accuracy']
    )
    model.fit(
        train_seq, epochs=cfg["phase2_epochs"],
        validation_data=val_seq,
        callbacks=get_callbacks(model_name, 2, fold, cfg),
        class_weight=class_weight_dict, verbose=1
    )
    return model


def evaluate_on_fold(model, val_paths, val_labels, model_name, fold, cfg,
                     print_report=False):
    """Runs inference, computes all metrics, returns result dict."""
    val_seq = FoldSequence(
        val_paths, val_labels,
        batch_size=cfg["batch_size"], img_size=cfg["img_size"],
        augment=False, num_classes=cfg["num_classes"]
    )
    y_true  = val_labels
    y_proba = model.predict(val_seq, verbose=0)
    y_pred  = np.argmax(y_proba, axis=1)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(
            to_categorical(y_true, cfg["num_classes"]),
            y_proba, multi_class='ovr', average='weighted')
    except Exception:
        auc = float('nan')

    if print_report:
        print(classification_report(y_true, y_pred,
              target_names=cfg["class_names"], zero_division=0))

    return {
        "model"    : model_name,
        "fold"     : fold,
        "accuracy" : float(acc),
        "precision": float(prec),
        "recall"   : float(rec),
        "f1"       : float(f1),
        "auc"      : float(auc),
        "y_proba"  : y_proba,
        "y_true"   : y_true,
        "y_pred"   : y_pred
    }

print("All functions defined. Ready for k-fold loop.")

## Cell 7 — Main K-Fold Loop
Runs **5 folds x 3 models = 15 training runs** + 5 ensemble evaluations.

**Resume-safe:** if Colab disconnects, re-run Cells 1–6 then this cell. Already-saved folds are skipped automatically.

In [ ]:
MODEL_NAMES      = ['mobilenetv2', 'resnet50', 'efficientnetb0']
all_fold_results = []   # one dict per (model, fold)
ensemble_results = []   # one dict per fold

skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True,
                      random_state=CFG["seed"])

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(all_paths, all_labels)):

    fold = fold_idx + 1
    print(f"\n{'='*62}")
    print(f"  FOLD {fold} / {CFG['n_folds']}  "
          f"(train={len(train_idx)}, val={len(val_idx)})")
    print(f"{'='*62}")

    # Class weights for this fold
    fold_labels = all_labels[train_idx]
    unique_cls  = np.unique(fold_labels)
    cw_array    = compute_class_weight('balanced', classes=unique_cls,
                                       y=fold_labels)
    class_weight_dict = {int(c): float(w)
                         for c, w in zip(unique_cls, cw_array)}

    # Generators
    train_seq = FoldSequence(
        all_paths[train_idx], all_labels[train_idx],
        batch_size=CFG["batch_size"], img_size=CFG["img_size"],
        augment=True, num_classes=CFG["num_classes"], seed=CFG["seed"]
    )
    val_seq = FoldSequence(
        all_paths[val_idx], all_labels[val_idx],
        batch_size=CFG["batch_size"], img_size=CFG["img_size"],
        augment=False, num_classes=CFG["num_classes"]
    )

    fold_probas = {}

    for model_name in MODEL_NAMES:
        final_path   = os.path.join(CFG["save_dir"],
                           f"{model_name}_fold{fold}_final.keras")
        metrics_path = os.path.join(CFG["save_dir"],
                           f"{model_name}_fold{fold}_metrics.json")

        # RESUME: skip if already done
        if os.path.exists(final_path) and os.path.exists(metrics_path):
            print(f"\n  [SKIP] {model_name} fold {fold} already done - loading saved data")
            with open(metrics_path) as mf:
                saved = json.load(mf)
            all_fold_results.append(saved)
            m = tf.keras.models.load_model(final_path)
            tmp_seq = FoldSequence(
                all_paths[val_idx], all_labels[val_idx],
                batch_size=CFG["batch_size"], img_size=CFG["img_size"],
                augment=False, num_classes=CFG["num_classes"]
            )
            fold_probas[model_name] = m.predict(tmp_seq, verbose=0)
            del m; gc.collect(); tf.keras.backend.clear_session()
            continue

        # Train
        print(f"\n  >>> {model_name.upper()} - Fold {fold}")
        model = train_one_model(model_name, fold, train_seq, val_seq,
                                class_weight_dict, CFG)

        # Evaluate
        result = evaluate_on_fold(model, all_paths[val_idx],
                                  all_labels[val_idx],
                                  model_name, fold, CFG, print_report=True)
        fold_probas[model_name] = result["y_proba"]

        print(f"  [{model_name} | Fold {fold}]  "
              f"Acc={result['accuracy']*100:.2f}%  "
              f"F1={result['f1']*100:.2f}%  "
              f"AUC={result['auc']:.4f}")

        # Save model
        model.save(final_path)
        print(f"  Saved -> {final_path}")

        # Save metrics JSON
        safe = {k: v for k, v in result.items()
                if k not in ('y_proba', 'y_true', 'y_pred')}
        with open(metrics_path, 'w') as mf:
            json.dump(safe, mf, indent=2)
        all_fold_results.append(safe)

        del model; gc.collect(); tf.keras.backend.clear_session()

    # Ensemble for this fold
    print(f"\n  >>> Ensemble - Fold {fold}")
    y_true_fold = all_labels[val_idx]

    # Simple average ensemble
    proba_avg = (fold_probas['mobilenetv2'] +
                 fold_probas['resnet50'] +
                 fold_probas['efficientnetb0']) / 3.0

    # Weighted ensemble (by individual accuracy)
    fold_accs = {mn: next(r['accuracy'] for r in all_fold_results
                          if r['model'] == mn and r['fold'] == fold)
                 for mn in MODEL_NAMES}
    total_acc = sum(fold_accs.values())
    proba_wt  = sum((fold_accs[mn] / total_acc) * fold_probas[mn]
                    for mn in MODEL_NAMES)

    for label, proba in [('Ensemble_SimpleAvg', proba_avg),
                          ('Ensemble_Weighted',  proba_wt)]:
        y_pred_ens = np.argmax(proba, axis=1)
        ens_acc    = accuracy_score(y_true_fold, y_pred_ens)
        ens_prec   = precision_score(y_true_fold, y_pred_ens,
                                     average='weighted', zero_division=0)
        ens_rec    = recall_score(y_true_fold, y_pred_ens,
                                  average='weighted', zero_division=0)
        ens_f1     = f1_score(y_true_fold, y_pred_ens,
                              average='weighted', zero_division=0)
        try:
            ens_auc = roc_auc_score(
                to_categorical(y_true_fold, CFG["num_classes"]),
                proba, multi_class='ovr', average='weighted')
        except Exception:
            ens_auc = float('nan')

        print(f"  [{label} | Fold {fold}]  "
              f"Acc={ens_acc*100:.2f}%  F1={ens_f1*100:.2f}%  "
              f"AUC={ens_auc:.4f}")

        ensemble_results.append({
            "model"    : label, "fold": fold,
            "accuracy" : float(ens_acc), "precision": float(ens_prec),
            "recall"   : float(ens_rec), "f1": float(ens_f1),
            "auc"      : float(ens_auc)
        })

    np.save(os.path.join(CFG["save_dir"], f"fold{fold}_avg_proba.npy"), proba_avg)
    np.save(os.path.join(CFG["save_dir"], f"fold{fold}_wt_proba.npy"),  proba_wt)

print(f"\n{'='*62}")
print("  ALL 5 FOLDS COMPLETE")
print(f"{'='*62}")

## Cell 8 — Summary Report
Mean ± Std across all 5 folds for every model and ensemble.

In [ ]:
all_model_labels     = MODEL_NAMES + ['Ensemble_SimpleAvg', 'Ensemble_Weighted']
all_results_combined = all_fold_results + ensemble_results

print(f"\n{'='*75}")
print("  5-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*75}")
print(f"  {'Model':<26} {'Acc Mean':>10}  {'Acc Std':>8}  "
      f"{'F1 Mean':>9}  {'AUC Mean':>10}")
print(f"  {'-'*70}")

summary_rows = []

for mn in all_model_labels:
    rows = [r for r in all_results_combined if r['model'] == mn]
    if not rows:
        continue
    accs  = [r['accuracy']  for r in rows]
    f1s   = [r['f1']        for r in rows]
    precs = [r['precision'] for r in rows]
    recs  = [r['recall']    for r in rows]
    aucs  = [r['auc']       for r in rows]

    row = {
        "model"    : mn,
        "acc_mean" : float(np.mean(accs)),  "acc_std" : float(np.std(accs)),
        "f1_mean"  : float(np.mean(f1s)),   "f1_std"  : float(np.std(f1s)),
        "prec_mean": float(np.mean(precs)), "rec_mean": float(np.mean(recs)),
        "auc_mean" : float(np.mean(aucs)),  "auc_std" : float(np.std(aucs)),
        "per_fold_acc": accs, "per_fold_f1": f1s
    }
    summary_rows.append(row)

    print(f"  {mn:<26} "
          f"{row['acc_mean']*100:>8.2f}%  "
          f"+-{row['acc_std']*100:>5.2f}%  "
          f"{row['f1_mean']*100:>8.2f}%  "
          f"{row['auc_mean']:>10.4f}")

print(f"{'='*75}")

with open(os.path.join(CFG["save_dir"], "kfold_summary.json"), 'w') as f:
    json.dump({
        "class_names"      : CFG["class_names"],
        "per_fold_results" : all_results_combined,
        "summary"          : summary_rows
    }, f, indent=2)

print("\nSummary JSON saved to Drive.")

## Cell 9 — Visualisations
Box plot, per-fold trend line, mean ± std bar chart, and confusion matrices.

In [ ]:
folds      = list(range(1, CFG["n_folds"] + 1))
plot_names = MODEL_NAMES + ['Ensemble_SimpleAvg', 'Ensemble_Weighted']
colors     = ['#5B8CDA', '#E07B54', '#56A882', '#A97FCC', '#D4855A']

# 1. Box plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('5-Fold Cross-Validation — Accuracy & F1 Distribution',
             fontsize=13, fontweight='bold')

acc_data, f1_data = [], []
for mn in plot_names:
    rows = sorted([r for r in all_results_combined if r['model'] == mn],
                  key=lambda x: x['fold'])
    acc_data.append([r['accuracy'] * 100 for r in rows])
    f1_data.append([r['f1']        * 100 for r in rows])

bp1 = axes[0].boxplot(acc_data, labels=plot_names, patch_artist=True)
bp2 = axes[1].boxplot(f1_data,  labels=plot_names, patch_artist=True)
for bp in [bp1, bp2]:
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col); patch.set_alpha(0.75)

for ax, title, ylabel in [
    (axes[0], 'Accuracy across 5 folds', 'Accuracy (%)'),
    (axes[1], 'F1 Score across 5 folds', 'F1 Score (%)')]:
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.set_ylim(80, 100); ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(CFG["save_dir"], 'kfold_boxplot.png'), dpi=150)
plt.show()

# 2. Per-fold accuracy trend
fig, ax = plt.subplots(figsize=(10, 5))
for mn, col in zip(plot_names, colors):
    rows      = sorted([r for r in all_results_combined if r['model'] == mn],
                       key=lambda x: x['fold'])
    fold_accs = [r['accuracy'] * 100 for r in rows]
    ls = '--' if 'Ensemble' in mn else '-'
    lw = 2.5  if 'Ensemble' in mn else 1.8
    ax.plot(folds, fold_accs, marker='o', label=mn,
            color=col, linestyle=ls, linewidth=lw)

ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('Validation Accuracy (%)', fontsize=11)
ax.set_title('Per-Fold Accuracy — All Models & Ensembles',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.set_xticks(folds)
ax.set_ylim(80, 100); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG["save_dir"], 'kfold_fold_trend.png'), dpi=150)
plt.show()

# 3. Mean +/- Std bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x         = np.arange(len(summary_rows))
bw        = 0.35
acc_means = [r['acc_mean'] * 100 for r in summary_rows]
f1_means  = [r['f1_mean']  * 100 for r in summary_rows]
acc_stds  = [r['acc_std']  * 100 for r in summary_rows]
f1_stds   = [r['f1_std']   * 100 for r in summary_rows]
labels_b  = [r['model']         for r in summary_rows]

b1 = ax.bar(x - bw/2, acc_means, bw, yerr=acc_stds,
            label='Accuracy', color='#5B8CDA', alpha=0.85, capsize=4)
b2 = ax.bar(x + bw/2, f1_means,  bw, yerr=f1_stds,
            label='F1 Score',  color='#E07B54', alpha=0.85, capsize=4)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{bar.get_height():.1f}',
            ha='center', va='bottom', fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(labels_b, rotation=15, ha='right')
ax.set_ylim(80, 104); ax.set_ylabel('Score (%)')
ax.set_title('Mean +/- Std — 5-Fold Cross-Validation',
             fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG["save_dir"], 'kfold_mean_std_bar.png'), dpi=150)
plt.show()

# 4. Confusion matrix for best fold per model
print("\nGenerating confusion matrices for best fold per model...")
skf2 = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True,
                       random_state=CFG["seed"])
fold_val_indices = {fi+1: va_i for fi, (_, va_i)
                    in enumerate(skf2.split(all_paths, all_labels))}

for mn in MODEL_NAMES:
    rows = sorted([r for r in all_fold_results if r['model'] == mn],
                  key=lambda x: x['accuracy'], reverse=True)
    if not rows:
        continue
    best_fold      = rows[0]['fold']
    best_model_path = os.path.join(CFG["save_dir"],
                          f"{mn}_fold{best_fold}_final.keras")
    if not os.path.exists(best_model_path):
        print(f"  Skipping {mn} — model file not found"); continue

    val_idx_best = fold_val_indices[best_fold]
    m   = tf.keras.models.load_model(best_model_path)
    vs  = FoldSequence(
        all_paths[val_idx_best], all_labels[val_idx_best],
        batch_size=CFG["batch_size"], img_size=CFG["img_size"],
        augment=False, num_classes=CFG["num_classes"]
    )
    y_true_b  = all_labels[val_idx_best]
    y_proba_b = m.predict(vs, verbose=0)
    y_pred_b  = np.argmax(y_proba_b, axis=1)
    del m; gc.collect(); tf.keras.backend.clear_session()

    cm = confusion_matrix(y_true_b, y_pred_b)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CFG["class_names"],
                yticklabels=CFG["class_names"], ax=ax)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True',      fontsize=11)
    ax.set_title(f'{mn} — Best Fold ({best_fold}) Confusion Matrix',
                 fontsize=12, fontweight='bold')
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.tight_layout()
    save_cm = os.path.join(CFG["save_dir"], f"{mn}_best_fold_cm.png")
    plt.savefig(save_cm, dpi=150); plt.show()
    print(f"  CM saved -> {save_cm}")

print("\nAll plots saved to Drive.")

## Cell 10 — Final Report Table
Copy this output into your thesis / research paper.

In [ ]:
print("\n" + "=" * 68)
print("  RESULTS TABLE — 5-FOLD CROSS-VALIDATION")
print("=" * 68)
print(f"  {'Model':<26} {'Accuracy':>16}   {'F1 Score':>14}   {'AUC':>8}")
print(f"  {'-'*65}")

for row in summary_rows:
    print(f"  {row['model']:<26} "
          f"{row['acc_mean']*100:>6.2f}% +/-{row['acc_std']*100:>4.2f}%   "
          f"{row['f1_mean']*100:>5.2f}% +/-{row['f1_std']*100:>4.2f}%   "
          f"{row['auc_mean']:>8.4f}")

print(f"\n{'='*68}")
print("""
  How to write this in your paper:
  ----------------------------------
  We evaluated the proposed nail disorder detection system using
  5-fold stratified cross-validation on a dataset of 4,147 images
  across 6 classes. The ensemble model achieved a mean accuracy of
  XX.XX% +/- Y.YY% and a weighted F1-score of XX.XX% +/- Y.YY%,
  demonstrating consistent performance across all folds and
  outperforming all individual base models.
""")